# Planet NDVI Calculation workflow

In this notebook, we take the imagery downloaded from Planet in the previous notebook, and process it into a NDVI table. We join the NDVI values to `inspection_id` in the SBCFD inspections datasets, so every inspection in Santa Barbara County that has every transpired will have it's own NDVI statistics.

The general workflow is...
1. Make all 0 values NaN.
    * When Planet clips imagery according to a given AOI, it makes all masked portions 0, rather than NA. We do this step to prevent the 0's from affecting the statistics calculations.
2. Calculate NDVI for every image.
3. Aggregate images into monthly mosaics.
4. Run the images on the parcels to calculate NDVI statistics.

In [1]:
import os
import shutil
import sys

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
import rasterio.merge
import rioxarray as rxr
from exactextract import exact_extract

from rioxarray.merge import merge_arrays
from shapely.geometry import box

sys.path.append("../utils")

nov_data_root = os.path.join(
    "/"
    "data",
    "wildfire_prep",
    "full_planet",
    "unzipped",
    "2020",
    "nov", 
    "files"
)

# general location of tabular data for this project (not satellite data)
capstone_data_root = "/capstone/wildfire_prep/data"

# dataset containing polygons to aggregate/perform NDVI statistics on
master_geometries = gpd.read_file('/capstone/wildfire_prep/data/PUZZLE_PIECES/inspections_master_training_geometries.geojson')



## Create filepath lists by month per year

These are used to iterate through every month in every year in the dataset.

In [2]:
month_sea = ["month_12", "month_11", "month_10", "month_09", "month_08", "month_07", "month_06", "month_05", "month_04", "month_03", "month_02", "month_01"]
month_name = ["dec", "nov", "oct", "sep", "aug", "jul", "jun", "may", "apr", "mar", "feb", "jan"]
year_sea = ["2019", "2020", "2021", "2022", "2023"]


## Function Creation

Here we create functions for...
1. 0 -> NaN
2. NDVI calculation
3. mosaicking
4. by parcel stats calculation

#### Make all 0's NaN

In [3]:
def masking(fps, output_path, year, month):

    # declare filepath vars
    year_month_output = f'{output_path}/{year}/{month}' # overall output folder appended with given year and month
    ndvi_root = "/data/wildfire_prep/full_planet/ndvi" # path to check if ndvi files exist
    mosaic_fp = f"/data/wildfire_prep/full_planet/mosaic/{year}/{month}/mosaic_{month}_{year}.tif" # do same for mosaic files

    # make a folder for the given year and month if it doesn't already exist
    os.makedirs(year_month_output, exist_ok=True)
    
    # for every filepath
    for fp in fps:

        # construct the output filepath/filename
        full_output = f"{year_month_output}/{fp[56:-36]}.tif"

        # and if neither the masked imagery, NDVI imagery, or corresponding mosaic exist
        if not os.path.exists(f"{ndvi_root}/{year}/{month}/ndvi_{fp[56:-36]}.tif") and not os.path.exists(full_output) and not os.path.exists(mosaic_fp):

            # source the imagery from the filepath
            with rasterio.open(fp) as src:
                profile = src.profile # extract the metadata from it
                layers = src.read().astype(np.float32) # and read in the imagery

                # then mask it
                layers[layers == 0] = np.nan

            # update the metadata in accordance with the newly masked metadata
            profile.update(dtype=rasterio.float32, count=layers.shape[0])
        
            # and write it out to the specified output filepath
            with rasterio.open(full_output, "w", **profile) as dst:
                dst.write(layers)

#### Calculate NDVI

In [4]:
def make_ndvi_tifs(fp_list, year, month): 

    # declare filepath vars
    ndvi_path_root = "/data/wildfire_prep/full_planet/ndvi" # broader filepath to write out to
    mosaic_fp = f"/data/wildfire_prep/full_planet/mosaic/{year}/{month}/mosaic_{month}_{year}.tif" # where to check for the mosaic

    # append selected year and month 
    full_dir = f'{ndvi_path_root}/{year}/{month}'

    # for every filepath
    for fp in fp_list:
        

        # make file paths and related vars
        file_name = f"ndvi_{fp[48:-4]}.tif" 
        output_fp = f'{full_dir}/{file_name}'

        # check if wgs filepath exists, make it if it doesn't
        if not os.path.exists(full_dir):
            os.makedirs(full_dir)

        # and if neither the ndvi calculation neither the mosaic exist
        if not os.path.exists(output_fp) and not os.path.exists(mosaic_fp):

            # Open image file
            # Load red band - note all PlanetScope 4-band images have band order BGRN
            with rasterio.open(fp) as src:
                band_red = src.read(3)

            # do same for near infrared
            with rasterio.open(fp) as src:
                band_nir = src.read(4)
            


            # Allow division by zero
            np.seterr(divide='ignore', invalid='ignore')

            # Calculate NDVI
            ndvi = (band_nir.astype(float) - band_red.astype(float)) / (band_nir + band_red)




            # Set spatial characteristics of the output object to mirror the input
            kwargs = src.meta
            kwargs.update(
                dtype=rasterio.float32,
                count = 1)

            # Create the file
            with rasterio.open(output_fp, 'w', **kwargs) as dst:
                dst.write_band(1, ndvi.astype(rasterio.float32))

        
        # delete every masked file after it's purpose has been fulfilled
        # not needed if you have enough storage space
        if os.path.exists(output_fp) & os.path.exists(fp):
            os.remove(fp)

    

#### Make mosaic

In [5]:
def make_mosaic(fps, output_folder, month, year):


  # Custom merge function to ignore NaNs
  def custom_merge_ignore_nan(old_data, new_data, old_nodata, new_nodata, index=None, roff=None, coff=None):
      valid_values = np.array([old_data, new_data])
      old_data[:] = np.nanmax(valid_values, axis=0)  # Compute max while ignoring NaNs


  # check if the mosaic already exists, if it doesn't
  if not os.path.exists(f"{output_folder}/mosaic_{month}_{year}.tif"):

    # reopen all ndvi rasters and append them to a list
    datasets = []
    for file in fps:
      ds = rxr.open_rasterio(file, masked=True)
      datasets.append(ds)

    

    # then mosaic the rasters together with the custom function
    merged_raster = merge_arrays(datasets, method=custom_merge_ignore_nan)

    # make the output folder if it doesn't exist yet
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # and save out the merged raster to it
    merged_raster.squeeze().rio.to_raster(f'{output_folder}/mosaic_{month}_{year}.tif')

#### Calculate NDVI statistics for a given parcel

In [6]:
def get_ndvi_value(tif, geometries, join_id = 'inspection_id'):
    
    intersect_count = 0

    # First, open the NDVI raster to get its extent
    print(type(tif))
    with rasterio.open(tif) as src:
        # Get the bounds of the raster
        raster_bounds = src.bounds
        # Create a polygon from the bounds
        raster_geometry = box(raster_bounds.left, raster_bounds.bottom, 
                            raster_bounds.right, raster_bounds.top)

        # Create a GeoDataFrame from the raster extent
        raster_extent = gpd.GeoDataFrame({'geometry': [raster_geometry]}, crs=src.crs)

        # Check if we need to reproject the extent
        if src.crs != "EPSG:3310":
            raster_extent = raster_extent.to_crs('EPSG:3310')


        # then get rid of records with no id
        ## but doesn't have much to filter
        geometries = geometries[geometries[join_id].notnull()]




    # Filter buffer geometries to only those that intersect with the raster extent
    intersecting_buffers = geometries[
        geometries.intersects(raster_extent.iloc[0].geometry)
    ]


    # Extract NDVI values only for the intersecting geometries
    # First ensure the CRS match
    if intersecting_buffers.crs != src.crs:
        intersecting_buffers = intersecting_buffers.to_crs(src.crs)

    

    # Use exactextract to calculate statistics for each geometry
    ndvi_stats = exact_extract(
        tif, # the raw unopened mosaic
        intersecting_buffers, # the processed parcel geometries
        ['mean', 'min', 'max'], # which statistics we will calculate
        include_cols=[join_id], # which other columns to include in the output
        output="pandas", # format of the output, in this case a DataFrame
    )

    # grab the total amount of intersecting buffers
    intersect_count = intersect_count + len(intersecting_buffers)
        


    print(f"length of original geometries: {len(geometries)}")
    print(f"total intersection polygons: {intersect_count}")
    print(f'length of end dataset for this month/year: {len(ndvi_stats)}')
    print("------------------------------\n")

    return ndvi_stats


## Full workflow begins here

We run through every function for every month, in every year. We then output a csv containing NDVI statistics for every inspection in Santa Barbara County from 2019 to 2023.

In [104]:
# redeclare the interatable variables
years = [2019, 2020, 2021, 2022, 2023]
month_name = ["dec", "nov", "oct", "sep", "aug", "jul", "jun", "may", "apr", "mar", "feb", "jan"]

month_num = list(reversed(range(1,13)))





response = input("Will run the full workflow. Execute? (yes/no): ")
if response.lower() == "yes":

    # for every year
    for year in years:

        # make an empty data frame with datatypes
        ndvi_csv = pd.DataFrame({
            'inspection_id': pd.Series(dtype = int), 
            'mean': pd.Series(dtype = float),
            'min': pd.Series(dtype = float),
            'max': pd.Series(dtype = float),
            'year': pd.Series(dtype = int),
            'month': pd.Series(dtype = int)
        })

        
        # make the unzipped file paths
        for month in month_name: 
            unzipped_list = []
            
            # init the target filepath for the raw imagery
            unzipped_fp = f'/data/wildfire_prep/full_planet/unzipped/{year}/{month}/files'

            # run through the target file path
            for root,dirs,files in os.walk(unzipped_fp, topdown=True):
                # and for every file
                for file in files:
                    # that matches the substring subset
                    if "AnalyticMS_SR_clip_reproject" in file:
                        # append it to a list
                        unzipped_list.append(f"{root}/{file}")

            # call the masking function for all months in a year
            print(f"making masks for {year}/{month}")
            masking(
                unzipped_list, # the list of filepaths built above
                '/data/wildfire_prep/full_planet/masked', # the target output path
                year, # target year
                month # target month
                )



        # make the masked fps, using the same process as for the raw files
        for month in month_name: 
            masked_list = []

            # init target filepath for masked imagery
            masked_fp = f'/data/wildfire_prep/full_planet/masked/{year}/{month}'

            # and run through each filepath
            for root,dirs,files in os.walk(masked_fp, topdown=True): 
                # append all masked filepaths to a list
                for file in files:
                    masked_list.append(f"{root}/{file}")

            # then run the ndvi calculation function for each month in year
            print(f"making ndvi for {year}/{month}")
            make_ndvi_tifs(masked_list, year, month)


        

        




        # ndvi file paths
        for month in month_name: 
            ndvi_list = []

            # init ndvi imagery fp
            ndvi_fp = f'/data/wildfire_prep/full_planet/ndvi/{year}/{month}'

            # run through each filepath
            for root,dirs,files in os.walk(ndvi_fp, topdown=True): 
                # for each filepath
                for file in files:
                    # append
                    ndvi_list.append(f"{root}/{file}")

            # mosaic every ndvi in that month with the mosaic function
            print(f"making mosaic for {year}/{month}")
            make_mosaic(
                ndvi_list, 
                f'/data/wildfire_prep/full_planet/mosaic/{year}/{month}',
                year=year, month=month
                )
            # remove whole ndvi directory for a given month in year
            shutil.rmtree(f'/data/wildfire_prep/full_planet/ndvi/{year}/{month}', ignore_errors=True)
            


        
                      
        # run through the iterable variables above, month name/number
        for month_abbrev, month_val in zip(month_name, month_num): 
            mosaic_list = []

            # init mosaic fp
            mosaic_fp = f'/data/wildfire_prep/full_planet/mosaic/{year}/{month_abbrev}'

            # for every fp
            for root,dirs,files in os.walk(mosaic_fp, topdown=True): 
                for file in files:

                    # append to list
                    mosaic_list.append(f"{root}/{file}")

            # then run the ndvi statistic calculation function
            print(f"making ndvi table for {year}/{month_abbrev}")
            curr_month = get_ndvi_value(
                mosaic_list[0], # the above list
                master_geometries[ # the parcel geometries
                        (master_geometries["year"] == int(year)) &  # subsetted for target year
                        (master_geometries["month"] == int(month_val)) # and month
                            ])
            
            # attach the associated the associated month and year to the NDVI stats for the current month
            # curr_month = curr_month.assign(year = year) 
            # curr_month = curr_month.assign(month = month_val)
            # then append it to the master df for the whole year
            ndvi_csv = pd.concat([ndvi_csv, curr_month])


        print(f'outputting {year} to csv...')

        # make inspection_id int
        ndvi_csv['inspection_id'] = ndvi_csv['inspection_id'].astype('Int64')

        # write out the csv for the whole year
        ndvi_csv.to_csv(f'/capstone/wildfire_prep/data/ndvi_csv/ndvi_parcels_{year}.csv')

        print(f'{year} csv complete.')



    print("Full NDVI workflow complete.")
else:
    print("Execution canceled.")

making masks for 2019/dec
making masks for 2019/nov
making masks for 2019/oct
making masks for 2019/sep
making masks for 2019/aug
making masks for 2019/jul
making masks for 2019/jun
making masks for 2019/may
making masks for 2019/apr
making masks for 2019/mar
making masks for 2019/feb
making masks for 2019/jan
making ndvi for 2019/dec
making ndvi for 2019/nov
making ndvi for 2019/oct
making ndvi for 2019/sep
making ndvi for 2019/aug
making ndvi for 2019/jul
making ndvi for 2019/jun
making ndvi for 2019/may
making ndvi for 2019/apr
making ndvi for 2019/mar
making ndvi for 2019/feb
making ndvi for 2019/jan
making mosaic for 2019/dec
making mosaic for 2019/nov
making mosaic for 2019/oct
making mosaic for 2019/sep
making mosaic for 2019/aug
making mosaic for 2019/jul
making mosaic for 2019/jun
making mosaic for 2019/may
making mosaic for 2019/apr
making mosaic for 2019/mar
making mosaic for 2019/feb
making mosaic for 2019/jan
making ndvi table for 2019/dec
<class 'str'>
length of original 

#### Concat every NDVI year into a single csv

In [109]:
if 'master_csv' in globals():
    del master_csv
    
master_csv = pd.DataFrame()

fp = '/capstone/wildfire_prep/data/ndvi_csv'

for root,dirs,files in os.walk(fp, topdown=True): 
    for file in files:
        if "old" not in file:
            master_csv = pd.concat([master_csv, pd.read_csv(f'{root}/{file}').drop(columns='Unnamed: 0')], ignore_index=True)

master_csv.to_csv(f'{fp}/ndvi_parcels_full.csv')

#### Attach it to the training parcel geometries

In [151]:
if 'full_ndvi' in globals():
    del full_ndvi
if 'geom_ndvi' in globals():
    del geom_ndvi

full_ndvi = pd.read_csv(f'{fp}/ndvi_parcels_full.csv').drop(columns= ['Unnamed: 0', 'year', 'month'])
geom_ndvi = master_geometries.merge(full_ndvi, on="inspection_id", how='left').drop_duplicates()

geom_ndvi.to_csv("/capstone/wildfire_prep/data/PUZZLE_PIECES/inspection_id_planet_ndvi")